*Submitted to the Google - The Gemma 4 Developer Agent Paper Track.*

This notebook is generated from `paper/PAPER.md` in the linked repository; the repository is the primary artifact and contains the environment, task generator, policies, graders and committed result files.

# Bounded Context Without Context Rewrites

## Flat context and an intact prefix cache for long-horizon agents

---

## Abstract

Long-horizon autonomous agents are bounded by context economics, not model capability. The standard
remedy — periodically compacting the context — keeps the window small by *rewriting* it, and a
rewrite invalidates the shared prefix a serving engine's automatic prefix cache depends on, re-billing
the whole prompt at full price. We describe a policy that keeps the window small *without* rewriting
the front: the system prompt and the turn-1 brief are pinned, and the window advances at the end. We
evaluate three policies over 60 turns of a bug-fixing task on one 12B model, measuring retention,
compute cost and prefix-cache reuse directly rather than inferring them. The bounded-anchor policy
holds a flat context while the alternatives grow to the ceiling or collapse on each compaction;
figures are in §5. We also document four ways such an evaluation produces a clean-looking result that
means nothing, and how each was caught.

## 1. Introduction

An agent that must remain coherent across hundreds of turns accumulates history. Two costs grow
with it: the memory footprint of the key/value cache, and the compute spent re-reading the context
each turn. The dominant response is **compaction** — periodically summarising or discarding the
middle of the conversation to return under the window.

Compaction shrinks the context but changes its *content*. With automatic prefix caching a cached
token costs a fraction of a fresh one, so a changed prefix means the prompt is re-processed (§4).

**Why this matters in deployment.** The Gemma 4 Developer Agent harness compacts the agent's history
at 14,336 tokens -- 2,048 below the output ceiling it also requests -- and acts on the prompt just
sent rather than prompt plus output, so it is one call late. The scorer refuses any call past 32,768
tokens, and with a LoRA adapter the KV cache on four L4s falls from ~46,000 to ~7,600. Context
capacity, not reasoning, is the binding constraint, and compaction is the field's response.

We therefore ask a narrow, measurable question: **can an agent keep a small context without
rewriting it?** Our policy pins an immutable front (system prompt and the turn-1 brief) and advances
a bounded window at the end. Nothing is summarised: the information-dense part of each turn is
kept, the superseded part dropped.

**Contributions.**
1. A context policy that bounds the working set while leaving the prefix shared with the previous
   turn intact (§3).
2. A direct, per-turn measurement of prefix-cache reuse, and a compute-cost model that prices a miss
   and a hit differently (§4).
3. A long-horizon evaluation that discriminates, with per-turn grading and a retention probe that
   cannot be satisfied by recency alone (§4.2).
4. A catalogue of four failure modes by which such an evaluation silently measures nothing (§6).

## 2. Related work

**Eviction and compression.** H2O [1], Scissorhands [2], TOVA [3] and SnapKV [4] shrink the cache by
scoring tokens and dropping the low-scoring ones, assuming survivors stay valid after eviction — an
assumption we do not share (§6, F4). Others repair positional integrity directly: CacheBlend [5]
recomputes part of the KV and re-encodes positions, CacheFocus [6] re-positions after pruning,
DSCache [7] stores pre-rotation keys, CacheGen [8] recomputes as a fallback. **We attempt none of
that**: we never rewrite the region the cache depends on.

**Attention sinks.** StreamingLLM [9] keeps sink tokens and a sliding window. The sink mechanism
[10] — a learned, first-token attractor — makes pinning an immutable front mechanically sensible.
Sinks stabilise a stream but are not a memory channel [11], which is why our probes test a *specific
fact* rather than fluency.

**Closest prior art.** SinkTrack [21] is nearest in *intent*: it anchors a model to its initial
context and reports context forgetting as a real failure of long generation. The mechanisms differ in
kind — SinkTrack is **model-level**, injecting features into the `<BOS>` representation, evaluated on
single-generation QA; ours is **policy-level**, deciding *which tokens the agent retains* and
measuring the serving-cost consequence over a long horizon. It is also a reason to expect a pinned
prefix to work: if the first token attracts attention, an agent that can keep its brief at the front
should.

**Agent memory.** MemGPT [12] pages context OS-style; the survey [13] taxonomises memory
operations. Both address what an agent should *store*, not the serving cost of rewriting it.

**Position and architecture.** *Lost in the Middle* [14] and *Found in the Middle* [15] establish
position-dependent context use. Gemma-4 makes it concrete: 40 of 48 layers use a 1024-token
local-attention window, so a turn-1 fact is unavailable to those layers regardless of policy (§6,
F4). Retention claims here concern **the tokens the agent retained**.

**Corrected results.** The §6 F3 result is a defect in our own benchmark; we report the corrected
task. A position paper argues venues need a refutations track [16], the literature documents leakage
and reporting error [17,18], and empirical-method work [19] shows comparisons favour the proposed
method — the bias F1 and F2 produce. We adopt artifact-review expectations [20].

## 3. Method

### 3.1 The three policies

Let the transcript be a sequence of turn blocks. Each turn appends the instruction and the reply.
The policies differ *only* in which blocks are retained when assembling the prompt.

- **`runtime` (bounded anchor).** Retains the turn-1 block and every *instruction* so far, verbatim,
  plus the single most recent *reply*. Instructions are small; a reply here is a full-file rewrite,
  redundant with the newest one. Oldest instructions drop first if the budget is exceeded, so growth
  is one instruction per turn and the front is never rewritten.
- **`linear` (grow and evict).** Retains the turn-1 block and as many recent instruction/reply pairs
  as fit the model's ceiling, dropping the **oldest** block on overflow. The naive baseline: a fixed
  window with no organisation.
- **`prune` (grow, evict, compact).** Identical to `linear`, plus a compaction every 30 turns keeping
  the anchor and the most recent turns. The status quo for long-running agents.

All three share the model, prompts, tools, turn budget and turn-1 anchor, so a difference is
attributable to the retention policy. `prune` is anchored deliberately: otherwise it would differ
from `linear` by two things at once and a collapse could not be attributed to either.

### 3.2 The task

`solution.py` is seeded with *n* faulty functions, and **turn 1 carries the complete bug report** —
for each bug, the exact wrong and required constant. Every later turn says only "Fix bug *k*." No
list is repeated and no reminder is given.

1. Every bug family is the **same shape**: a one-line body with a single wrong constant. An earlier
   version used a keyword default and a table lookup; both invited the model to *rewrite* the shape,
   and the rewrites failed with the brief one turn back. A periodic, deterministic failure is a task
   defect, not a retention result (§6, F3).
2. The required constants are **arbitrary and appear nowhere else**, so they cannot be re-derived.
3. Grading is **per-turn**: bug *k* is tested right after turn *k*, in a subprocess, so a fix applied
   and later clobbered by a rewrite is distinguishable from one never applied.

### 3.3 Two independent retention probes

- **Accumulated codes.** One fresh unguessable code is handed over in each turn and may not be
  written to the file until the final turn, when it must be returned as an ordered list. Scored
  0..*n*, so partial retention appears as a prefix gap.
- **Release gate.** On the final turn, one extra function must return a **separate** secret handed
  over at turn *n*/2. This value is deliberately *not* one of the codes, so it is independent of the
  first probe: a policy can recite the list and still ship a broken artifact. Verified independent:
  two simulated agents with identical bug and code scores receive opposite gate verdicts.

**Why the gate discriminates, checked before the run and confirmed against it.** The mid-session
secret is handed over at turn *n*/2 and is required only at the end, so it is recoverable exactly
when the turn *n*/2 instruction is still in the prompt. Both policies anchor the turn-1 brief, so the
brief's survival is not the discriminator; **window depth** is.

The two policies diverge in growth rate by roughly an order of magnitude, measured live: `linear`
adds ~1,248 tokens per turn (a full instruction/reply pair) while `runtime` adds ~159 (one
instruction plus a single superseded reply). Against a 12k ceiling, `linear` therefore reaches the
ceiling at about **turn 9** and evicts from then on, holding only the most recent ~4 turn blocks —
so by the end its window reaches back ~4 turns and the turn-30 secret is long gone. `runtime` holds
every instruction and finishes at ~9.9k tokens, still under budget, with the mid-session instruction
reachable. This was computed against the real tokenizer before the run and then observed in the run
itself.

## 4. Measurement

### 4.1 Prefix-cache reuse

Per turn we record the longest common **token** prefix between the current prompt and the previous
one — exactly what an automatic prefix cache would reuse. Cost is reported in **raw compute units,
not currency**: a fresh token costs 1.0 and a cache-reused token costs an assumed multiplier (0.1),
so

```
cost_units = miss_tokens · 1.0 + hit_tokens · HIT_MULT

```

and the no-cache baseline is `total_tokens · 1.0`. The multiplier is one named constant so the
assumption is visible and re-runnable. This makes the cache claim falsifiable: if the policies do
not separate on hit rate, the argument does not hold.

### 4.2 Metrics collected

Retention (per-turn success, final accuracy, fixes applied then lost, code recall and ordering, gate
verdict); cost (hit rate, miss tokens, cost units, no-cache baseline, saving ratio); context (first,
last, peak, growth); latency (TTFT, wall per turn, decode throughput); memory (KV peak, total
prefill). All derive from data the loop already collects, so one run yields the whole table rather
than one run per question.

### 4.3 Fairness gate

The stopping rule is the **turn count**, not wall-clock time. This is load-bearing: a policy that
keeps a smaller context is faster per turn *by construction*, so a time-based stop would let it
complete more turns and win on volume rather than on policy. Time is retained only as an emergency
break, so that an interrupted run grades what it finished instead of being discarded.

Because that break can fire at different turn counts per arm, the comparison script does **not**
report unequal arms as a difference and does **not** discard the run. It reconciles to the
**common prefix** — the turns both arms actually completed — re-summing the cost, context and
latency figures over that prefix. The measurements are per-turn and indexed by turn, so the turns
captured on both sides are the same turns, which is what comparability requires. It refuses only
when fewer than two turns are comparable.

One asymmetry is deliberate: the **retention** counts are not reconstructed, because the artifact's
state at turn *n* cannot be recovered from a later snapshot. A run that stops early is therefore
compared on cost and context over the shared prefix, and its retention numbers describe only the
turns it reached.

## 5. Results

*(to be completed from the run in flight; every arm reports the full §4.2 table, and no number is
quoted until the final-turn probes have been checked on both arms)*

## 6. Failure modes of long-horizon benchmarks

Each of the following produced, or would have produced, a clean-looking table that measured nothing.

**F1 — shared ceiling.** Capping two policies at the same limit makes both converge on "whatever
fits": with one shared 12k cap both arms reached ~9.9k with identical latency (4180 vs 4308 ms).
The cap, not the policy, set the working set. Each policy now gets a budget sized to its own claim.

**F2 — re-derivable probes.** A rule the model re-applies every turn can be inferred from recent
turns without the original ever being seen, so both arms pass and the probe is vacuous. Only a value
stated once and never repeated can discriminate.

**F3 — periodic failures are task defects.** Two of six bug families failed on *every* instance from
turn 2 — one turn after the brief, so memory was not in question: the families had unusual shapes the
model rewrote. A turn-indexed task must be checked for periodic failure before its losses count.

**F4 — architectural confound.** Gemma-4 uses local attention in 40 of 48 layers with a 1024-token
window, so beyond 1024 tokens a turn-1 fact is invisible to those layers regardless of policy. Any
claim about retaining an early fact must be stated against that background.

## 7. Limitations

One model, one task family; the task is synthetic by design so retention is measurable. The cost model
uses an assumed hit multiplier rather than a billing measurement. The gate probe is a single value:
it establishes that a mid-session fact survived, not a rate.

**Model choice.** The reported run uses Gemma-4-12B bf16, which loads on one accelerator with no
quantization confound; the competition checkpoint (`gemma-4-31b-it-qat-w4a16-ct`) is 4-bit and would
add a quantization variable to a retention measurement. The policy is model-independent — a decision
about retained blocks, implemented outside the model — so another checkpoint changes the numbers, not
the mechanism.

## 8. Conclusion

Keeping a context small and keeping it *stable* are different problems, and the usual solution to the
first destroys the second. A policy that pins an immutable front and rolls the window at the end
bounds the working set without invalidating the prefix the previous turn already paid for. We have
made the corresponding claim measurable rather than asserted, and we have documented the ways such a
measurement can silently fail.

---

## Reproduction

```
python3 tools/run_compare.py benchmarks/results/incremental_*_linear_*.json \
                               benchmarks/results/incremental_*_runtime_*.json

```

Environment, task generator, policies and graders are in this repository; the command above runs
against committed result files. The three policies are selected by `CCAI_ARM` (`runfix3.sh` sets
all three); the retention gate is opt-in via `CCAI_GATE=1`; the compaction period by
`CCAI_PRUNE_EVERY`.

## References

[1] Zhang et al. *H2O: Heavy-Hitter Oracle for Efficient Generative Inference of LLMs.* NeurIPS 2023. arXiv:2306.14048
[2] Liu et al. *Scissorhands: Exploiting the Persistence of Importance Hypothesis.* NeurIPS 2023. arXiv:2305.17118
[3] Oren et al. *Transformers are Multi-State RNNs.* arXiv:2401.06104
[4] Li et al. *SnapKV: LLM Knows What You Are Looking For Before Generation.* NeurIPS 2024. arXiv:2404.14469
[5] Yao et al. *CacheBlend: Fast LLM Serving for RAG with Cached Knowledge Fusion.* EuroSys 2025. arXiv:2405.16444
[6] Lee, Park & Han. *CacheFocus: Dynamic Cache Re-Positioning for Efficient RAG.* arXiv:2502.11101
[7] Pang et al. *Decouple and Cache: KV Cache Construction for Streaming Video.* arXiv:2605.01858
[8] Liu et al. *CacheGen: KV Cache Compression and Streaming for Fast LLM Serving.* ACM SIGCOMM 2024. arXiv:2310.07240
[9] Xiao et al. *Efficient Streaming Language Models with Attention Sinks.* ICLR 2024. arXiv:2309.17453
[10] Gu et al. *When Attention Sink Emerges in Language Models.* arXiv:2410.10781
[11] Cao, Zhang & Tang. *Separating Stream Stability from Long-Term Recall in Language Models.* arXiv:2609.07282
[12] Packer et al. *MemGPT: Towards LLMs as Operating Systems.* arXiv:2310.08560
[13] Zhang et al. *A Survey on the Memory Mechanism of LLM-based Agents.* arXiv:2404.13501
[14] Liu et al. *Lost in the Middle: How Language Models Use Long Contexts.* TACL 2024. arXiv:2307.03172
[15] Hsieh, Chuang et al. *Found in the Middle: Calibrating Positional Attention Bias Improves Long Context Utilization.* arXiv:2406.16008
[21] Liu, Chen & Wang. *SinkTrack: Attention Sink based Context Anchoring for Large Language Models.* ICLR 2026. arXiv:2604.10027  ← **verified against the index (authors + venue confirmed)**
[16] Schaeffer, Kazdan, Denisov-Blanch, Miranda, Gerstgrasser et al. *Position: Machine Learning Conferences Should Establish a "Refutations and Critiques" Track.* NeurIPS 2025 Position Paper Track (Oral). arXiv:2506.19882
[17] Kapoor & Narayanan. *Leakage and the reproducibility crisis in ML-based science.* Patterns 2023.
[18] *Systematic research errors in thousands of machine learning papers.* ACL 2023.
[19] Herrmann et al. *Why We Must Rethink Empirical Research in Machine Learning.* ICML 2024.
[20] ACM. *Artifact Review and Badging, v1.1.*

> ⚠️ Several of these entries came from research notes rather than a direct index lookup and are
> marked UNVERIFIED in the source material. arXiv identifiers in the `26xx` range are valid for
> 2026 (e.g. `2609.*` = September 2026). **Each citation's authors, venue and id must be confirmed
> against the index before submission** — an unverifiable citation is a defect to fix, not a fact
> to assert, and the [3],[6],[7],[11] entries in particular have unconfirmed author lists.

## Figure 1

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1120 420" width="1120" height="420">
<rect width="1120" height="420" fill="#ffffff"/>
<line x1="74.0" y1="334.0" x2="544.0" y2="334.0" stroke="#333" stroke-width="1"/>
<line x1="74.0" y1="34.0" x2="74.0" y2="334.0" stroke="#333" stroke-width="1"/>
<line x1="74.0" y1="334.0" x2="544.0" y2="334.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="66.0" y="338.0" font-size="11" fill="#555" text-anchor="end">0</text>
<line x1="74.0" y1="259.0" x2="544.0" y2="259.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="66.0" y="263.0" font-size="11" fill="#555" text-anchor="end">2500</text>
<line x1="74.0" y1="184.0" x2="544.0" y2="184.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="66.0" y="188.0" font-size="11" fill="#555" text-anchor="end">5000</text>
<line x1="74.0" y1="109.0" x2="544.0" y2="109.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="66.0" y="113.0" font-size="11" fill="#555" text-anchor="end">7500</text>
<line x1="74.0" y1="34.0" x2="544.0" y2="34.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="66.0" y="38.0" font-size="11" fill="#555" text-anchor="end">10000</text>
<text x="74.0" y="352.0" font-size="11" fill="#555" text-anchor="middle">0</text>
<text x="191.5" y="352.0" font-size="11" fill="#555" text-anchor="middle">14</text>
<text x="309.0" y="352.0" font-size="11" fill="#555" text-anchor="middle">29</text>
<text x="426.5" y="352.0" font-size="11" fill="#555" text-anchor="middle">44</text>
<text x="544.0" y="352.0" font-size="11" fill="#555" text-anchor="middle">59</text>
<text x="309.0" y="372.0" font-size="12" fill="#333" text-anchor="middle">turn</text>
<text x="24.0" y="184.0" font-size="12" fill="#333" text-anchor="middle" transform="rotate(-90 24.0 184.0)">prompt tokens (tokens)</text>
<text x="309.0" y="22.0" font-size="13" font-weight="600" fill="#111" text-anchor="middle">(a) context growth</text>
<polyline points="74.0,272.3 82.0,181.3 89.9,178.9 97.9,176.6 105.9,174.2 113.8,171.8 121.8,169.5 129.8,167.1 137.7,164.7 145.7,162.3 153.7,159.9 161.6,157.4 169.6,155.0 177.6,152.5 185.5,150.0 193.5,147.6 201.5,145.1 209.4,142.7 217.4,140.2 225.4,137.7 233.3,135.3 241.3,132.8 249.3,130.4 257.2,127.9 265.2,125.4 273.2,123.0 281.1,120.5 289.1,118.1 297.1,115.6 305.0,113.1 313.0,109.6 320.9,107.2 328.9,104.7 336.9,102.2 344.8,99.8 352.8,97.3 360.8,94.9 368.7,92.4 376.7,89.9 384.7,87.5 392.6,85.0 400.6,82.6 408.6,80.1 416.5,77.6 424.5,75.2 432.5,72.7 440.4,70.3 448.4,67.8 456.4,65.4 464.3,62.9 472.3,60.4 480.3,58.0 488.2,55.5 496.2,53.1 504.2,50.6 512.1,48.1 520.1,45.7 528.1,43.2 536.0,40.8 544.0,36.0" fill="none" stroke="#1b6ca8" stroke-width="2"/>
<line x1="600.0" y1="334.0" x2="1070.0" y2="334.0" stroke="#333" stroke-width="1"/>
<line x1="600.0" y1="34.0" x2="600.0" y2="334.0" stroke="#333" stroke-width="1"/>
<line x1="600.0" y1="334.0" x2="1070.0" y2="334.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="592.0" y="338.0" font-size="11" fill="#555" text-anchor="end">0</text>
<line x1="600.0" y1="259.0" x2="1070.0" y2="259.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="592.0" y="263.0" font-size="11" fill="#555" text-anchor="end">25</text>
<line x1="600.0" y1="184.0" x2="1070.0" y2="184.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="592.0" y="188.0" font-size="11" fill="#555" text-anchor="end">50</text>
<line x1="600.0" y1="109.0" x2="1070.0" y2="109.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="592.0" y="113.0" font-size="11" fill="#555" text-anchor="end">75</text>
<line x1="600.0" y1="34.0" x2="1070.0" y2="34.0" stroke="#e6e6e6" stroke-width="1"/>
<text x="592.0" y="38.0" font-size="11" fill="#555" text-anchor="end">100</text>
<text x="600.0" y="352.0" font-size="11" fill="#555" text-anchor="middle">0</text>
<text x="717.5" y="352.0" font-size="11" fill="#555" text-anchor="middle">14</text>
<text x="835.0" y="352.0" font-size="11" fill="#555" text-anchor="middle">29</text>
<text x="952.5" y="352.0" font-size="11" fill="#555" text-anchor="middle">44</text>
<text x="1070.0" y="352.0" font-size="11" fill="#555" text-anchor="middle">59</text>
<text x="835.0" y="372.0" font-size="12" fill="#333" text-anchor="middle">turn</text>
<text x="550.0" y="184.0" font-size="12" fill="#333" text-anchor="middle" transform="rotate(-90 550.0 184.0)">reused (%)</text>
<text x="835.0" y="22.0" font-size="13" font-weight="600" fill="#111" text-anchor="middle">(b) prefix-cache reuse</text>
<polyline points="600.0,334.0 608.0,213.1 615.9,111.4 623.9,110.2 631.9,109.1 639.8,108.0 647.8,106.9 655.8,105.9 663.7,104.9 671.7,103.9 679.7,103.1 687.6,102.2 695.6,101.3 703.6,100.4 711.5,99.5 719.5,98.6 727.5,97.8 735.4,97.0 743.4,96.2 751.4,95.4 759.3,94.6 767.3,93.9 775.3,93.2 783.2,92.5 791.2,91.8 799.2,91.1 807.1,90.5 815.1,89.8 823.1,89.2 831.0,88.6 839.0,89.1 846.9,88.5 854.9,86.6 862.9,86.0 870.8,85.5 878.8,84.9 886.8,84.4 894.7,83.9 902.7,83.4 910.7,82.9 918.6,82.4 926.6,81.9 934.6,81.5 942.5,81.0 950.5,80.6 958.5,80.1 966.4,79.7 974.4,79.3 982.4,78.9 990.3,78.5 998.3,78.1 1006.3,77.7 1014.2,77.3 1022.2,76.9 1030.2,76.5 1038.1,76.2 1046.1,75.8 1054.1,75.4 1062.0,75.1 1070.0,76.8" fill="none" stroke="#1b6ca8" stroke-width="2"/>
<line x1="920" y1="16" x2="942" y2="16" stroke="#1b6ca8" stroke-width="3"/>
<text x="948" y="20" font-size="12" fill="#333">runtime</text>
</svg>

Growth (a) and prefix reuse (b), per turn, from the same per-turn records the tables are generated from. Source: `paper/figs/trajectory.svg`, regenerable with `tools/make_figures.py`.